In [1]:
import polars as pl
import numpy as np
from pathlib import Path
from catboost import CatBoostRanker, Pool
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = Path("/kaggle/input/datasets/fedorfesenko/train-interactions")
WORK = Path("/kaggle/working")
WORK.mkdir(parents=True, exist_ok=True)

In [2]:
df_train = pl.read_parquet(DATA_DIR / "cf_partial_all.parquet")
df_val = pl.read_parquet(DATA_DIR / "val_dataset.parquet")
test_dataset = pl.read_parquet(DATA_DIR / "test_dataset.parquet")
df_test_orig = pl.read_parquet(DATA_DIR / "df_test.parquet")
df_ranking_val = pl.read_parquet(DATA_DIR / "df_ranking_val.parquet")
df_ranking_train = pl.read_parquet(DATA_DIR / "df_ranking_train.parquet")

candgen_train_v2 = pl.read_parquet(DATA_DIR / "candgen_features_train_v2.parquet")
candgen_val_v2 = pl.read_parquet(DATA_DIR / "candgen_features_val_v2.parquet")
candgen_test_v2 = pl.read_parquet(DATA_DIR / "candgen_features_test_v2.parquet")

print(f"df_train: {df_train.shape}")
print(f"df_val: {df_val.shape}")
print(f"test_dataset: {test_dataset.shape}")
print(f"candgen_train_v2: {candgen_train_v2.shape}")

df_train: (5429559, 69)
df_val: (323736, 74)
test_dataset: (292826, 72)
candgen_train_v2: (15769472, 17)


In [3]:
key_cols = ["user_id", "product_id"]

candgen_train_v2 = candgen_train_v2.unique(subset=key_cols, keep="first")

df_train_v8 = df_train.join(
    df_ranking_train.select(["user_id", "product_id", "request_id", "product_category"]).unique(),
    on=["user_id", "product_id", "request_id"], how="left"
)
print(f"df_train_v8 после product_category: {df_train_v8.shape}")

df_train_v8 = df_train_v8.join(candgen_train_v2, on=key_cols, how="left")
print(f"df_train_v8 после candgen: {df_train_v8.shape}")

df_train_v8 = df_train_v8.with_columns([
    pl.from_epoch(pl.col("timestamp"), time_unit="s").dt.hour().alias("hour_of_day"),
    pl.from_epoch(pl.col("timestamp"), time_unit="s").dt.weekday().alias("day_of_week"),
])

print(f"df_train_v8: {df_train_v8.shape}")
print(f"product_category null: {df_train_v8['product_category'].null_count()}")
print(f"candgen nulls: {df_train_v8.select(['als_score','v6_score','als_cosine']).null_count()}")

df_train_v8 после product_category: (5429559, 70)
df_train_v8 после candgen: (5429559, 85)
df_train_v8: (5429559, 87)
product_category null: 0
candgen nulls: shape: (1, 3)
┌───────────┬──────────┬────────────┐
│ als_score ┆ v6_score ┆ als_cosine │
│ ---       ┆ ---      ┆ ---        │
│ u32       ┆ u32      ┆ u32        │
╞═══════════╪══════════╪════════════╡
│ 0         ┆ 0        ┆ 0          │
└───────────┴──────────┴────────────┘


In [4]:
candgen_val_v2 = candgen_val_v2.unique(subset=key_cols, keep="first")
candgen_val_v2 = candgen_val_v2.drop(["als_score", "v5_score", "v6_score"])

df_val_v8 = df_val.join(candgen_val_v2, on=key_cols, how="left")
print(f"df_val_v8 после candgen: {df_val_v8.shape}")

if "product_category" not in df_val_v8.columns:
    cat_val = df_ranking_val.select(["user_id", "product_id", "request_id", "product_category"]).unique()
    df_val_v8 = df_val_v8.join(cat_val, on=["user_id", "product_id", "request_id"], how="left")

print(f"df_val_v8: {df_val_v8.shape}")

df_val_v8 после candgen: (323736, 86)
df_val_v8: (323736, 87)


In [5]:
key_cols_3 = ["user_id", "product_id", "request_id"]

candgen_test_v2 = pl.read_parquet(DATA_DIR / "candgen_features_test_v2.parquet")
candgen_test_v2 = candgen_test_v2.unique(subset=key_cols, keep="first")
candgen_test_v2 = candgen_test_v2.drop(["als_score", "v5_score", "v6_score"])

test_features_dedup = test_dataset.unique(subset=key_cols_3, keep="first")
cols_to_drop = [c for c in ["index", "timestamp", "source_type", "city_name"] if c in test_features_dedup.columns]
test_features_dedup = test_features_dedup.drop(cols_to_drop)

df_test_v8 = df_test_orig.join(test_features_dedup, on=key_cols_3, how="left", suffix="_features")
drop_right = [c for c in df_test_v8.columns if c.endswith("_right") or c.endswith("_features")]
if drop_right:
    df_test_v8 = df_test_v8.drop(drop_right)

df_test_v8 = df_test_v8.join(candgen_test_v2, on=key_cols, how="left")

if "hour_of_day" not in df_test_v8.columns:
    df_test_v8 = df_test_v8.with_columns([
        pl.from_epoch(pl.col("timestamp"), time_unit="s").dt.hour().alias("hour_of_day"),
        pl.from_epoch(pl.col("timestamp"), time_unit="s").dt.weekday().alias("day_of_week"),
    ])

print(f"df_test_v8: {df_test_v8.shape}")
print(f"unique index: {df_test_v8['index'].n_unique()}")

df_test_v8: (292826, 88)
unique index: 292826


In [6]:
BASE_NUM = [
    "n_views_all", "n_clicks_all", "n_cart_updates_all", "n_purchases_all",
    "n_views_30d", "n_clicks_30d", "n_cart_updates_30d", "n_purchases_30d",
    "n_views_7d", "n_clicks_7d", "n_cart_updates_7d", "n_purchases_7d",
    "n_views_1d", "n_clicks_1d", "n_cart_updates_1d", "n_purchases_1d",
    "days_since_first_action", "days_since_last_action",
    "item_n_views_all", "item_n_clicks_all", "item_n_cart_updates_all", "item_n_purchases_all",
    "item_n_views_30d", "item_n_clicks_30d", "item_n_cart_updates_30d", "item_n_purchases_30d",
    "item_n_views_7d", "item_n_clicks_7d", "item_n_cart_updates_7d", "item_n_purchases_7d",
    "item_n_views_1d", "item_n_clicks_1d", "item_n_cart_updates_1d", "item_n_purchases_1d",
    "item_n_unique_users_all", "item_n_unique_users_30d", "item_n_unique_users_7d", "item_n_unique_users_1d",
    "item_days_since_first_view", "item_days_since_last_view",
    "ui_n_views_all", "ui_n_clicks_all", "ui_n_cart_updates_all", "ui_n_purchases_all",
    "ui_days_since_last_action", "ui_days_since_last_cart_update", "ui_days_since_last_purchase",
    "uc_n_views_all", "uc_n_clicks_all", "uc_n_cart_updates_all", "uc_n_purchases_all",
    "uc_n_views_30d", "uc_n_clicks_30d", "uc_n_cart_updates_30d", "uc_n_purchases_30d",
    "uc_n_views_7d", "uc_n_clicks_7d", "uc_n_cart_updates_7d", "uc_n_purchases_7d",
    "uc_days_since_last_purchase",
    "als_score", "v5_score", "v6_score",
    "hour_of_day", "day_of_week",
    "als_user_norm", "als_item_norm", "als_cosine", "als_norms_prod",
    "v5_user_norm",  "v5_item_norm",  "v5_cosine",  "v5_norms_prod",
    "v6_user_norm",  "v6_item_norm",  "v6_cosine",  "v6_norms_prod",
]
CAT_FEATURES = ["source_type", "city_name", "product_category"]
FEATURES = BASE_NUM + CAT_FEATURES

print(f"NUM: {len(BASE_NUM)}, CAT: {len(CAT_FEATURES)}, total: {len(FEATURES)}")

for name, df in [("train", df_train_v8), ("val", df_val_v8), ("test", df_test_v8)]:
    missing = [f for f in FEATURES if f not in df.columns]
    print(f"missing in {name}: {missing}")

def prepare_pool(df, features, cat_features, target_col="target", group_col="request_id"):
    df = df.sort(group_col)
    X = df.select(features).to_pandas()
    for c in features:
        if c not in cat_features:
            X[c] = X[c].astype("float32")
    for c in cat_features:
        X[c] = X[c].fillna("unknown").astype(str)
    if target_col in df.columns:
        y = df[target_col].to_numpy().astype("float32")
    else:
        y = np.zeros(len(df), dtype="float32")
    group_id = df[group_col].to_numpy()
    return Pool(data=X, label=y, group_id=group_id, cat_features=cat_features), df

train_pool_v8, df_train_sorted_v8 = prepare_pool(df_train_v8, FEATURES, CAT_FEATURES)
val_pool_v8, df_val_sorted_v8 = prepare_pool(df_val_v8, FEATURES, CAT_FEATURES)
test_pool_v8, df_test_sorted_v8 = prepare_pool(df_test_v8, FEATURES, CAT_FEATURES)

print(f"train: {train_pool_v8.num_row()}")
print(f"val: {val_pool_v8.num_row()}")
print(f"test: {test_pool_v8.num_row()}")

NUM: 77, CAT: 3, total: 80
missing in train: []
missing in val: []
missing in test: []

Preparing pools...
train: 5429559
val: 323736
test: 292826


In [7]:
model_v8 = CatBoostRanker(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    loss_function="YetiRank",
    eval_metric="NDCG:top=10",
    task_type="CPU",
    thread_count=-1,
    verbose=100,
    random_seed=42,
    allow_writing_files=False,
)

model_v8.fit(
    train_pool_v8,
    eval_set=val_pool_v8,
    early_stopping_rounds=50,
    use_best_model=True,
    verbose=100,
)

Groupwise loss function. OneHotMaxSize set to 10
0:	test: 0.8480228	best: 0.8480228 (0)	total: 17.9s	remaining: 4h 58m 28s
100:	test: 0.8811454	best: 0.8813383 (95)	total: 27m 17s	remaining: 4h 2m 59s
200:	test: 0.8836753	best: 0.8842430 (165)	total: 51m 44s	remaining: 3h 25m 42s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.8842429731
bestIteration = 165

Shrink model to first 166 iterations.


CatBoostRanker(allow_writing_files=False, depth=8, eval_metric='NDCG:top=10', iterations=1000, learning_rate=0.05, loss_function='YetiRank', random_seed=42, task_type='CPU', verbose=100)

In [8]:
test_preds_v8 = model_v8.predict(test_pool_v8)
print(f"test_preds_v8: {test_preds_v8.shape}")

df_sub_v8 = pl.DataFrame({
    "index": df_test_sorted_v8["index"].to_numpy(),
    "request_id": df_test_sorted_v8["request_id"].to_numpy(),
    "score": test_preds_v8.astype("float64"),
})
df_sub_v8_sorted = df_sub_v8.sort(["request_id", "score"], descending=[False, True])
df_sub_v8_final = df_sub_v8_sorted.select(["index", "request_id"])
df_sub_v8_final.write_csv("submission_v8.csv")

print(f"\nsubmission_v8.csv: {df_sub_v8_final.shape}")
print(f"unique index: {df_sub_v8_final['index'].n_unique()}")

test_preds_v8: (292826,)

submission_v8.csv: (292826, 2)
unique index: 292826
